In [18]:
%pip install langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from dotenv import load_dotenv
import os

C:\Users\Lenovo yoga\PycharmProjects\darazscraper\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
documents = [
    Document(page_content="Car repair and vehicle servicing guide. Learn how to maintain your automobile.", metadata={"topic": "vehicles"}),
    Document(page_content="Python is a popular programming language for data science and AI applications.", metadata={"topic": "programming"}),
    Document(page_content="The history of sedan cars and how automotive engineering evolved over time.", metadata={"topic": "vehicles"}),
    Document(page_content="Machine learning models learn patterns from large datasets automatically.", metadata={"topic": "AI"}),
    Document(page_content="How to cook Italian pasta from scratch with fresh ingredients.", metadata={"topic": "cooking"}),
    Document(page_content="Truck maintenance and diesel engine repair for commercial vehicles.", metadata={"topic": "vehicles"}),
    Document(page_content="Deep learning uses neural networks with many layers for complex tasks.", metadata={"topic": "AI"}),
    Document(page_content="Baking bread at home: sourdough starter and fermentation tips.", metadata={"topic": "cooking"}),
    Document(page_content="Electric vehicle battery technology and charging infrastructure guide.", metadata={"topic": "vehicles"}),
    Document(page_content="Natural language processing enables computers to understand human text.", metadata={"topic": "AI"}),
]

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5286.64it/s]


In [9]:
load_dotenv()
URL = os.getenv("URL")
API_KEY = os.getenv("APIKEY")

vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    api_key=API_KEY,
    url=URL,
    collection_name="search_vectors",
)

print(f"length of the documents is {len(documents)}")

length of the documents is 10


In [10]:
#the k value must be moderate neither too small nor too big
query = "automobile maintenance"
print(f"Query: '{query}'")
print(f"(No document uses the exact word 'automobile maintenance")

results = vectorstore.similarity_search(query=query,k=30)

for i,result in enumerate(results):
    print(f"{i+1},result: {result.metadata}")
    print(f"{i+1},result: {result.page_content}")

Query: 'automobile maintenance'
(No document uses the exact word 'automobile maintenance
1,result: {'topic': 'vehicles', '_id': '9f06bd1e-5c4c-4348-8d26-f5f22258e67c', '_collection_name': 'search_vectors'}
1,result: Truck maintenance and diesel engine repair for commercial vehicles.
2,result: {'topic': 'vehicles', '_id': '6b53d257-9a44-4678-b78f-3d435619b3b1', '_collection_name': 'search_vectors'}
2,result: Truck maintenance and diesel engine repair for commercial vehicles.
3,result: {'topic': 'vehicles', '_id': '71653473-7089-466a-b6e1-0064475d15c0', '_collection_name': 'search_vectors'}
3,result: Truck maintenance and diesel engine repair for commercial vehicles.
4,result: {'topic': 'vehicles', '_id': '8952e1e2-8e1f-46e0-92f0-27d254973210', '_collection_name': 'search_vectors'}
4,result: Truck maintenance and diesel engine repair for commercial vehicles.
5,result: {'topic': 'vehicles', '_id': '3204429e-2c8e-40d2-8bb4-a86cbe20d528', '_collection_name': 'search_vectors'}
5,result: Truc

**.as_retriever : can use with any vectore databse**

In [11]:
# This is the standard way to search in LangChain

retriever = vectorstore.as_retriever()

results_lang = retriever.invoke(query)
for i,result in enumerate(results_lang):
    print(f"metadata of result {i+1},is: {result.metadata}")
    print(f"content of result {i+1},is: {result.page_content[:80]}")


metadata of result 1,is: {'topic': 'vehicles', '_id': '9f06bd1e-5c4c-4348-8d26-f5f22258e67c', '_collection_name': 'search_vectors'}
content of result 1,is: Truck maintenance and diesel engine repair for commercial vehicles.
metadata of result 2,is: {'topic': 'vehicles', '_id': '6b53d257-9a44-4678-b78f-3d435619b3b1', '_collection_name': 'search_vectors'}
content of result 2,is: Truck maintenance and diesel engine repair for commercial vehicles.
metadata of result 3,is: {'topic': 'vehicles', '_id': '8952e1e2-8e1f-46e0-92f0-27d254973210', '_collection_name': 'search_vectors'}
content of result 3,is: Truck maintenance and diesel engine repair for commercial vehicles.
metadata of result 4,is: {'topic': 'vehicles', '_id': '71653473-7089-466a-b6e1-0064475d15c0', '_collection_name': 'search_vectors'}
content of result 4,is: Truck maintenance and diesel engine repair for commercial vehicles.


In [12]:
# ============================================================
# COMPARE: Different k Values
# ============================================================
# Let's search with k=2, k=5, and k=8 to see the difference.

query = "Tell me about artificial intelligence"
print(f"Query: '{query}'\n")

# Test 3 different k values
k_values = [2, 5, 8]

for k in k_values:
    # search_kwargs is a dictionary of search parameters
    # {"k": k} tells the retriever how many documents to return
    retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
    results = retriever_k.invoke(query)

    print(f"--- k={k} ({len(results)} results) ---")
    for i, doc in enumerate(results):
        # [:60] shows first 60 characters of each document
        print(f"  #{i + 1} [{doc.metadata['topic']}]: {doc.page_content[:60]}...")
    print()

Query: 'Tell me about artificial intelligence'

--- k=2 (2 results) ---
  #1 [AI]: Deep learning uses neural networks with many layers for comp...
  #2 [AI]: Deep learning uses neural networks with many layers for comp...

--- k=5 (5 results) ---
  #1 [AI]: Deep learning uses neural networks with many layers for comp...
  #2 [AI]: Deep learning uses neural networks with many layers for comp...
  #3 [AI]: Deep learning uses neural networks with many layers for comp...
  #4 [AI]: Deep learning uses neural networks with many layers for comp...
  #5 [AI]: Deep learning uses neural networks with many layers for comp...

--- k=8 (8 results) ---
  #1 [AI]: Deep learning uses neural networks with many layers for comp...
  #2 [AI]: Deep learning uses neural networks with many layers for comp...
  #3 [AI]: Deep learning uses neural networks with many layers for comp...
  #4 [AI]: Deep learning uses neural networks with many layers for comp...
  #5 [AI]: Deep learning uses neural networks with ma

**you only want results that are actually relevant?The score threshold filters out low-quality matches.**

In [13]:
#Threshold-> use to set a limit of similarity e.g stop when similarity drops form 0.70 stop showing those results.

threshold_retriever = vectorstore.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {"score_threshold":0.5},
)

results_threshold = threshold_retriever.invoke(query)
for i,result in enumerate(results_threshold):
    print(f"metadata of result {i+1},is: {result.metadata}")
    print(f"content of result {i+1},is: {result.page_content[:70]}...")

if len(results) < 8:
    print(f"\nOnly {len(results)} results passed the threshold!")
    print(f"The irrelevant cooking/vehicle docs were filtered out.")


metadata of result 1,is: {'topic': 'AI', '_id': 'daf35498-d11c-4f97-931c-eb720531169e', '_collection_name': 'search_vectors'}
content of result 1,is: Deep learning uses neural networks with many layers for complex tasks....
metadata of result 2,is: {'topic': 'AI', '_id': '0a12857b-fe7f-48c5-b03d-f842f4efe90d', '_collection_name': 'search_vectors'}
content of result 2,is: Deep learning uses neural networks with many layers for complex tasks....
metadata of result 3,is: {'topic': 'AI', '_id': '786b6ce6-dfe7-4214-8bc1-345fa3ed7da7', '_collection_name': 'search_vectors'}
content of result 3,is: Deep learning uses neural networks with many layers for complex tasks....
metadata of result 4,is: {'topic': 'AI', '_id': '04378ac4-56e8-4e16-a5bd-ce6e76932a61', '_collection_name': 'search_vectors'}
content of result 4,is: Deep learning uses neural networks with many layers for complex tasks....


**MMR(Maximum Marginal Relevance)**

***We want to be relevant while give diverse response to user,for that we use mmr->solve redundant results problem***

In [17]:
#Search without MMR
relevance_search = vectorstore.similarity_search(query,k=5)

for i,search in enumerate(relevance_search):
    print(f"metadata of result {i+1},is: {search.metadata}")
    print(f"content of result {i+1},is: {search.page_content}...")


##Search with MMR
mmr_search = vectorstore.max_marginal_relevance_search(
    query,
    k=5,      #how many final results
    fetch_k=8, #How many docs to consider
    lambda_mult = 0.5,   #keep moderate relevance and diversity
)

for i,search2 in enumerate(mmr_search):
    print(f"metadata of result {i+1},is: {search2.metadata}")
    print(f"content of result {i+1},is: {search2.page_content}...")

# Show the difference
standard_ids = [doc.metadata["id"] for doc in relevance_search]
mmr_ids = [doc.metadata["id"] for doc in mmr_search]
print(f"\nStandard doc IDs: {standard_ids}")
print(f"MMR doc IDs:      {mmr_ids}")
print(f"\nMMR brings in MORE diverse documents!")

metadata of result 1,is: {'topic': 'AI', '_id': '4aecf7ee-fa75-4828-b4ce-e61fab36a4d8', '_collection_name': 'search_vectors'}
content of result 1,is: Deep learning uses neural networks with many layers for complex tasks....
metadata of result 2,is: {'topic': 'AI', '_id': 'daf35498-d11c-4f97-931c-eb720531169e', '_collection_name': 'search_vectors'}
content of result 2,is: Deep learning uses neural networks with many layers for complex tasks....
metadata of result 3,is: {'topic': 'AI', '_id': '786b6ce6-dfe7-4214-8bc1-345fa3ed7da7', '_collection_name': 'search_vectors'}
content of result 3,is: Deep learning uses neural networks with many layers for complex tasks....
metadata of result 4,is: {'topic': 'AI', '_id': '0a12857b-fe7f-48c5-b03d-f842f4efe90d', '_collection_name': 'search_vectors'}
content of result 4,is: Deep learning uses neural networks with many layers for complex tasks....
metadata of result 5,is: {'topic': 'AI', '_id': '04378ac4-56e8-4e16-a5bd-ce6e76932a61', '_collection_nam

KeyError: 'id'

In [ ]:
# ============================================================
# DEMO: MMR with Different Lambda Values
# ============================================================
# Let's see how lambda_mult affects the balance.

query = "What is NLP?"
print(f"Query: '{query}'\n")

# Test 3 different lambda values
lambda_values = [
    (1.0, "Pure relevance (same as standard)"),
    (0.5, "Balanced (recommended)"),
    (0.2, "Heavy diversity"),
]

for lam, description in lambda_values:
    results = vectorstore.max_marginal_relevance_search(
        query, k=4, fetch_k=8, lambda_mult=lam,
    )

    # Collect the document IDs to see which docs were picked
    ids = [doc.metadata["id"] for doc in results]

    print(f"Lambda={lam} — {description}")
    print(f"  Doc IDs returned: {ids}")
    for i, doc in enumerate(results):
        # [:60] shows first 60 characters
        print(f"    #{i + 1}: {doc.page_content[:60]}...")
    print()

In [ ]:
# ============================================================
# BUILD: MMR Retriever (production-ready)
# ============================================================
# This is how you'd use MMR in a real RAG system.

# Standard retriever (similarity only)
standard_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# MMR retriever (relevance + diversity)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,              # Return 4 documents
        "fetch_k": 8,        # Consider 8 candidates
        "lambda_mult": 0.5,  # Balanced relevance/diversity
    },
)

print("Two retrievers created:")
print("  1. Standard: search_type='similarity', k=4")
print("  2. MMR:      search_type='mmr', k=4, fetch_k=8, lambda=0.5")
print()

# Compare them
query = "What is NLP?"

standard_docs = standard_retriever.invoke(query)
mmr_docs = mmr_retriever.invoke(query)

print(f"Query: '{query}'\n")

print("Standard retriever:")
for i, doc in enumerate(standard_docs):
    print(f"  #{i + 1}: {doc.page_content[:65]}...")

print("\nMMR retriever:")
for i, doc in enumerate(mmr_docs):
    print(f"  #{i + 1}: {doc.page_content[:65]}...")

print(f"\nMMR retriever gives more diverse results!")
print(f"Use this in your RAG chain for better answers.")